# Notebook B: entity annotation of the multi-turn datasets, with web search, built on the paper's own pipeline

This notebook labels every assistant turn in the `longfact-multiturn-<model>` datasets, using the code in
[obalcells/hallucination_probes](https://github.com/obalcells/hallucination_probes) `annotation_pipeline/` directly.
- **Prompt:** the repo's `entity_annotation.prompt` (the paper's prompt), filled in by the repo's `format_prompt`.
  As in `run.py`, the instruction is `conversation[-2]` (that turn's user message) and the completion is `conversation[-1]`.
  The repo's comment says this is how multi-turn data would be handled.
- **Settings:** temperature 0 and `max_tokens` 8192.
- **Parsing:** the repo's `parse_and_validate_json`, into the `AnnotatedSpan` schema.
  The labels are `Supported`, `Not Supported` and `Insufficient Information`; the repo counts Insufficient Information as hallucinated.
- **Span positions:** the repo's `assign_span_positions`, so each annotation is `{span, label, verification_note, index}`.

**Web search.** The paper used Claude Sonnet 4 with Anthropic's built-in web search. Here the judge runs in one of two ways:
It tries Adaption first and falls back to HF for any answer where Adaption fails (error or output that can't be parsed).
Each row's `annotator` field records which one labelled it.
- **Adaption (first):** an Adaption model with **Exa's hosted web-search MCP server** attached as an MCP tool.
  Adaption makes the search calls itself, and the Exa server needs no key (it has a free rate limit).
- **HF (fallback):** gpt-oss-120b on HF credits. This code runs each `web_search` the judge asks for through DuckDuckGo (`ddgs`), at most 15 per answer (repo `run.py`).

Output goes to `<input-repo>-annotated`, uploaded file by file; Run all again to resume.
The last cell gives the **hallucination rate per model and per turn**, on the conversations that every model has.
Runs on CPU in a local Jupyter notebook (Python 3.10+). Files are written next to the notebook (`repo/`, `work/`).

Keys are read from environment variables, or asked for (hidden input) if they are not set:
- **`HF_TOKEN`**: a **write** token of the account in `HF_USER`.
- **`ADAPTION_API_KEY`**: for the Adaption judge.
- **`HF_INFERENCE_TOKEN`** (optional, for the HF fallback): a token with "Make calls to Inference Providers" from the account whose credits should pay.
  If it is not set, `HF_TOKEN` is used.

In [1]:
!pip install -q -U huggingface_hub openai ddgs rouge-score "pydantic>=2.6"
# download the paper's repo (no git needed)
import io, os, urllib.request, zipfile
if not os.path.isdir("repo"):
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(
        "https://github.com/obalcells/hallucination_probes/archive/refs/heads/main.zip").read())).extractall(".")
    os.rename("hallucination_probes-main", "repo")
print("repo ready:", sorted(os.listdir("repo")))

repo ready: ['.gitignore', 'LICENSE', 'README.md', 'annotation_pipeline', 'configs', 'demo', 'probe', 'pyproject.toml', 'requirements.txt', 'train_data.yaml', 'utils', 'uv.lock']


In [2]:
# ---------------- settings ----------------
SRC_USER = "shreshthamodi02"                # account that holds the multi-turn datasets (public, read only)
HF_USER = "shreshtha2002"                   # account that receives the annotated datasets (HF_TOKEN is its write token)
INPUT_REPOS = ["longfact-multiturn-aya-expanse-8b-q8-0",
               "longfact-multiturn-llama3.1-8b-instruct-q8-0"]
BACKENDS = ["hf-mcp", "hf"]                 # tried in order for each answer:
                                            #   "hf-mcp"   = HF Responses API, Exa web-search MCP run by HF's router
                                            #   "hf"       = HF chat + our DuckDuckGo search loop (fallback)
                                            #   "adaption" = Adaption + Exa MCP (gave 504s)
EXA_MCP = "https://mcp.exa.ai/mcp?tools=web_search_exa"   # Exa's hosted MCP server, keyless with a free rate limit
JUDGES = {"hf-mcp": "openai/gpt-oss-120b:groq", "hf": "openai/gpt-oss-120b:groq", "adaption": "deepseek-v4-flash"}
TEMPERATURE, MAX_TOKENS, MAX_SEARCHES = 0.0, 8192, 15    # repo: annotate.py / run.py (search cap enforced for "hf" only)
WORKERS = 3                                 # answers annotated in parallel (lower it if searches start failing)
FILES_LIMIT = 5                             # 5 files x 10 conversations x 3 turns = first 50 conversations per model; None = all 200

In [3]:
import sys, subprocess, importlib, site
print("kernel python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "install", "rouge-score", "nltk", "absl-py", "ddgs", "openai",
                "huggingface_hub", "pydantic>=2.6"], check=True)
site.addsitedir(site.getusersitepackages())   # in case pip installed to your user folder
importlib.invalidate_caches()
import rouge_score, ddgs, openai; print("all installed OK")

kernel python: \\?\C:\Users\Shreshtha\AppData\Roaming\jupyterlab-desktop\jlab_server\python.exe
all installed OK


In [4]:
# ---------------- the repo's annotation code ----------------
import ast, importlib.util, json, logging, os, re, sys, time
from typing import List
sys.path.insert(0, "repo")
from utils.parsing import parse_and_validate_json
import utils.string_utils as string_utils
from utils.string_utils import try_matching_span_in_text
from rouge_score import rouge_scorer

spec = importlib.util.spec_from_file_location("data_models", "repo/annotation_pipeline/data_models.py")
data_models = importlib.util.module_from_spec(spec); spec.loader.exec_module(data_models)
AnnotatedSpan = data_models.AnnotatedSpan

# format_prompt and assign_span_positions, taken verbatim from annotate.py (importing the file itself would need Anthropic's safety-tooling)
repo_fns = {"List": List, "AnnotatedSpan": AnnotatedSpan, "try_matching_span_in_text": try_matching_span_in_text,
            "logger": logging.getLogger("annotate")}
for node in ast.parse(open("repo/annotation_pipeline/annotate.py", encoding="utf-8").read()).body:
    if isinstance(node, ast.FunctionDef) and node.name in ("format_prompt", "assign_span_positions"):
        exec(compile(ast.Module([node], []), "annotate.py", "exec"), repo_fns)
format_prompt, assign_span_positions = repo_fns["format_prompt"], repo_fns["assign_span_positions"]
PROMPT = open("repo/annotation_pipeline/entity_annotation.prompt", encoding="utf-8").read().strip()   # as in annotate.py

class UnicodeTokenizer:          # ROUGE's default tokenizer deletes Devanagari; split on spaces/punctuation instead
    def tokenize(self, text):
        return [t for t in re.split(r"[\s.,;:!?\"'()\[\]\-–—।]+", text.lower()) if t]
string_utils.ROUGE_SCORER = rouge_scorer.RougeScorer(["rougeL"], tokenizer=UnicodeTokenizer())
print(PROMPT[:300], "...")

You are a fact-checker. Your task is to fact-check a model's completion to some given instruction.

Here's the instruction:
<instruction>{instruction}</instruction>

Here's the completion you'll have to fact-check:
<completion>{completion}</completion>

Focus on verifying:
- People (names, affiliati ...


In [5]:
# ---------------- judge + web search ----------------
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from ddgs import DDGS
from huggingface_hub import HfApi
from openai import OpenAI

from getpass import getpass
def secret(name):
    return os.environ.get(name) or getpass(f"{name}: ")

HF_TOKEN =                            # write token of HF_USER (datasets)
HF_INFERENCE_TOKEN = os.environ.get("HF_INFERENCE_TOKEN") or HF_TOKEN   # account whose credits pay the judge (if different)
api = HfApi(token=HF_TOKEN)
USER = HF_USER
me = api.whoami()
assert USER in [me["name"]] + [o["name"] for o in me.get("orgs", [])], (
    f"HF_TOKEN belongs to '{me['name']}', which cannot upload to '{USER}'. Use a write token created while logged in as '{USER}'.")
log = lambda m: print(time.strftime("%H:%M:%S"), m, flush=True)
clients = {}
if "adaption" in BACKENDS:
    clients["adaption"] = OpenAI(base_url="https://api.prod.adaptionlabs.ai/api/v1", api_key=secret("ADAPTION_API_KEY"), max_retries=3, timeout=600)
if {"hf", "hf-mcp"} & set(BACKENDS):
    clients["hf"] = clients["hf-mcp"] = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_INFERENCE_TOKEN,
                                               max_retries=6, timeout=600)

SEARCH = {"name": "web_search", "description": "Search the web. Returns the top results (title, URL, snippet).",
          "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}

def web_search(query):
    for attempt in range(4):
        try:
            hits = DDGS().text(query, max_results=5)
            return "\n\n".join(f"{h.get('title')}\n{h.get('href')}\n{h.get('body')}" for h in hits) or "No results."
        except Exception:
            time.sleep(3 * 2 ** attempt)
    return "Search failed."

def run_tool(args, n):
    if n > MAX_SEARCHES:
        return "Search limit reached. Give your final answer now."
    return web_search(json.loads(args or "{}").get("query", ""))

def judge(user_prompt, backend):
    """One user message with the repo's prompt; the judge may call web_search up to MAX_SEARCHES times."""
    n = 0
    client, model = clients[backend], JUDGES[backend]
    if backend == "hf":                                        # Chat Completions tool calling
        msgs = [{"role": "user", "content": user_prompt}]
        while True:
            m = client.chat.completions.create(model=model, messages=msgs, tools=[{"type": "function", "function": SEARCH}],
                                               tool_choice="auto" if n < MAX_SEARCHES else "none",
                                               temperature=TEMPERATURE, max_tokens=MAX_TOKENS).choices[0].message
            if not m.tool_calls:
                return m.content or ""
            msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [t.model_dump() for t in m.tool_calls]})
            for t in m.tool_calls:
                n += 1
                msgs.append({"role": "tool", "tool_call_id": t.id, "content": run_tool(t.function.arguments, n)})
    r = client.responses.create(                             # "hf-mcp" / "adaption": the server runs the MCP search calls itself
        model=model, input=[{"role": "user", "content": user_prompt}], temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS,
        tools=[{"type": "mcp", "server_label": "exa", "server_url": EXA_MCP, "allowed_tools": ["web_search_exa"],
                "require_approval": "never"}])
    SEARCHES.append(sum(o.type == "mcp_call" for o in r.output))
    return r.output_text

SEARCHES = []   # MCP searches per answer, for a sanity check in the log

def annotate(row):
    """Same steps as the repo's annotate_completion."""
    instruction, completion = row["conversation"][-2]["content"], row["conversation"][-1]["content"]
    for backend in BACKENDS:                                   # first backend; the next one if it fails
        for attempt in range(2):
            try:
                spans = parse_and_validate_json(judge(format_prompt(instruction, completion, PROMPT), backend),
                                                List[AnnotatedSpan], allow_partial=True)
                spans = assign_span_positions(spans, completion)
                return {**row, "annotations": [s.model_dump() for s in spans],
                        "annotator": f"{backend}/{JUDGES[backend]}+{'ddgs' if backend == 'hf' else 'exa-mcp'}"}
            except Exception as e:
                log(f"  {backend} attempt {attempt + 1} failed for {row['conversation_id']} turn {row['turn']}: {str(e)[:200]}")
    return {**row, "annotations": None, "annotator": None}   # every backend failed; left unlabelled

HF_TOKEN:  ········


In [6]:
# ---------------- web search through Exa's MCP server, called from this notebook ----------------
import json, urllib.request

def exa_mcp(query):
    # minimal MCP (streamable HTTP) client: initialize -> initialized -> tools/call web_search_exa
    H = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream",
         "User-Agent": "Mozilla/5.0 (hallucination-annotator)"}
    def rpc(body, sid=None):
        req = urllib.request.Request(EXA_MCP, data=json.dumps(body).encode(), headers={**H, **({"Mcp-Session-Id": sid} if sid else {})})
        with urllib.request.urlopen(req, timeout=60) as r:
            sid = r.headers.get("Mcp-Session-Id") or sid
            txt = r.read().decode("utf-8")
        data = [l[5:].strip() for l in txt.splitlines() if l.startswith("data:")]
        msg = json.loads(data[-1]) if data else (json.loads(txt) if txt.strip() else {})
        if "error" in msg:
            raise RuntimeError(msg["error"])
        return sid, msg
    sid, _ = rpc({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                  "params": {"protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "annotator", "version": "0.1"}}})
    rpc({"jsonrpc": "2.0", "method": "notifications/initialized"}, sid)
    _, msg = rpc({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
                  "params": {"name": "web_search_exa", "arguments": {"query": query, "numResults": 5}}}, sid)
    return "\n\n".join(c.get("text", "") for c in msg["result"]["content"])

def web_search(query):            # Exa MCP first; DuckDuckGo if Exa fails
    try:
        return exa_mcp(query)[:2500]          # keeps the judge's context (and your credits) small
    except Exception as e:
        log(f"  Exa MCP failed ({str(e)[:100]}); using DuckDuckGo")
    for attempt in range(3):
        try:
            hits = DDGS().text(query, max_results=5)
            return "\n\n".join(f"{h.get('title')}\n{h.get('href')}\n{h.get('body')}" for h in hits) or "No results."
        except Exception:
            time.sleep(3 * 2 ** attempt)
    return "Search failed."

BACKENDS = ["hf"]                 # HF chat completions + the web_search tool above (HF's own MCP feature gave 401)

def judge(user_prompt, backend="hf"):
    """The repo's prompt as one user message; the judge may call web_search up to MAX_SEARCHES times (repo run.py: 15)."""
    client, model, n = clients["hf"], JUDGES["hf"], 0
    msgs = [{"role": "user", "content": user_prompt}]
    while True:
        extra = {"tools": [{"type": "function", "function": SEARCH}]} if n < MAX_SEARCHES else {}   # at the cap: no tools, so it must answer
        m = client.chat.completions.create(model=model, messages=msgs, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                                           **extra).choices[0].message
        if not m.tool_calls or n >= MAX_SEARCHES:
            return m.content or ""
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [t.model_dump() for t in m.tool_calls]})
        for t in m.tool_calls:
            n += 1
            msgs.append({"role": "tool", "tool_call_id": t.id, "content": run_tool(t.function.arguments, n)})

def annotate(row):
    """Same steps as the repo's annotate_completion."""
    instruction, completion = row["conversation"][-2]["content"], row["conversation"][-1]["content"]
    for attempt in range(3):
        try:
            spans = parse_and_validate_json(judge(format_prompt(instruction, completion, PROMPT), "hf"),
                                            List[AnnotatedSpan], allow_partial=True)
            spans = assign_span_positions(spans, completion)
            return {**row, "annotations": [s.model_dump() for s in spans], "annotator": f"hf/{JUDGES['hf']}+exa-mcp"}
        except Exception as e:
            log(f"  attempt {attempt + 1} failed for {row['conversation_id']} turn {row['turn']}: {str(e)[:300]}")
    return {**row, "annotations": None, "annotator": None}

print(web_search("Chicago Tylenol murders 1982")[:500])

01:53:09   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
Chicago Tylenol murders - Wikipedia
https://en.wikipedia.org/wiki/Chicago_Tylenol_murders
The Chicago Tylenol murders were an unsolved series of poisoning deaths resulting from drug tampering in the Chicago metropolitan area in 1982. The victims consumed Tylenol branded acetaminophen (paracetamol) powder-filled capsules that had been adulterated with potassium cyanide. At least seven people died in the original poisonings, and there were several more deaths in subsequent copycat ...

The Chicago


In [7]:
def judge(user_prompt, backend="hf"):
    """The repo's prompt as one user message; the judge may call web_search up to MAX_SEARCHES times (repo run.py: 15)."""
    client, model, n, found = clients["hf"], JUDGES["hf"], 0, []
    tools = [{"type": "function", "function": SEARCH}]
    msgs = [{"role": "user", "content": user_prompt}]
    for _ in range(MAX_SEARCHES + 5):            # tools stay on (Groq rejects a tool call when tools are removed)
        m = client.chat.completions.create(model=model, messages=msgs, tools=tools, temperature=TEMPERATURE,
                                           max_tokens=MAX_TOKENS).choices[0].message
        if not m.tool_calls:
            return m.content or ""
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [t.model_dump() for t in m.tool_calls]})
        for t in m.tool_calls:
            n += 1
            result = run_tool(t.function.arguments, n)      # past the cap this returns "Search limit reached. Give your final answer now."
            if n <= MAX_SEARCHES:
                found.append(f"[search: {json.loads(t.function.arguments or '{}').get('query', '')}]\n{result}")
            msgs.append({"role": "tool", "tool_call_id": t.id, "content": result})
    # rare: it kept searching past the cap -> one last call without tools, with the results it already gathered
    final = (user_prompt + "\n\nWeb search results already gathered:\n\n" + "\n\n".join(found)
             + "\n\nNo more searches are possible. Return the JSON array now.")
    return client.chat.completions.create(model=model, messages=[{"role": "user", "content": final}],
                                          temperature=TEMPERATURE, max_tokens=MAX_TOKENS).choices[0].message.content or ""

In [8]:
INCLUDE_HISTORY = True   # False = repo behaviour (latest user message only)

def annotate(row):
    """Same steps as the repo's annotate_completion; for turns 2-3 the instruction also carries the earlier turns."""
    conv = row["conversation"]
    completion = conv[-1]["content"]
    if INCLUDE_HISTORY and len(conv) > 2:
        history = "\n\n".join(f"{m['role'].upper()}: {m['content']}" for m in conv[:-2])
        instruction = f"[Earlier conversation]\n{history}\n\n[Current request]\n{conv[-2]['content']}"
    else:
        instruction = conv[-2]["content"]
    for attempt in range(3):
        try:
            spans = parse_and_validate_json(judge(format_prompt(instruction, completion, PROMPT), "hf"),
                                            List[AnnotatedSpan], allow_partial=True)
            spans = assign_span_positions(spans, completion)   # positions are found only in the latest answer
            return {**row, "annotations": [s.model_dump() for s in spans],
                    "annotator": f"hf/{JUDGES['hf']}+exa-mcp" + ("+history" if INCLUDE_HISTORY else "")}
        except Exception as e:
            log(f"  attempt {attempt + 1} failed for {row['conversation_id']} turn {row['turn']}: {str(e)[:300]}")
    return {**row, "annotations": None, "annotator": None}

In [ ]:
# ---------------- annotate each file, save locally, upload (Run again to resume) ----------------
import urllib.request

def download(repo_id, filename):
    # plain urllib download (avoids the old-brotli crash inside huggingface_hub); public datasets need no token
    path = os.path.join("downloads", repo_id.replace("/", "__"), filename)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with urllib.request.urlopen(f"https://huggingface.co/datasets/{repo_id}/resolve/main/{filename}", timeout=120) as r, open(path, "wb") as fh:
        fh.write(r.read())
    return path

for name in INPUT_REPOS:
    src, dst = f"{SRC_USER}/{name}", f"{USER}/{name}-annotated"
    api.create_repo(dst, repo_type="dataset", private=False, exist_ok=True)
    done = set(api.list_repo_files(dst, repo_type="dataset"))
    files = sorted(f for f in api.list_repo_files(src, repo_type="dataset") if f.startswith("data/") and f.endswith(".jsonl"))
    files = files[:FILES_LIMIT] if FILES_LIMIT else files
    log(f"===== {src} -> https://huggingface.co/datasets/{dst}  ({len(files)} files, {len(done & set(files))} done)")
    for f in files:
        if f in done:
            continue
        t = time.time()
        rows = [json.loads(l) for l in open(download(src, f), encoding="utf-8")]
        with ThreadPoolExecutor(WORKERS) as pool:
            out = list(pool.map(annotate, rows))
        local = os.path.join("work", name, f)
        os.makedirs(os.path.dirname(local), exist_ok=True)
        with open(local, "w", encoding="utf-8") as fh:
            fh.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in out)
        api.upload_file(path_or_fileobj=local, path_in_repo=f, repo_id=dst, repo_type="dataset", commit_message=f"annotate {f}")
        labels = [a["label"] for r in out for a in r["annotations"] or [] if a["index"] is not None]
        log(f"  {f}: {len(out)} answers ({sum(r['annotations'] is None for r in out)} failed), {len(labels)} entities: "
            f"{labels.count('Supported')} S / {labels.count('Not Supported')} NS / {labels.count('Insufficient Information')} II "
            f"in {time.time() - t:.0f}s | by {dict(Counter(r['annotator'] for r in out))}")
log("All done.")

01:53:12 ===== shreshthamodi02/longfact-multiturn-aya-expanse-8b-q8-0 -> https://huggingface.co/datasets/shreshtha2002/longfact-multiturn-aya-expanse-8b-q8-0-annotated  (5 files, 0 done)
01:53:15   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:16   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:16   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:26   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:39   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:39   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:41   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:50   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:53   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:53:59   Exa MCP failed (HTTP Error 429: Too Many Requests); using DuckDuckGo
01:54:01   Ex

In [ ]:
# ---------------- hallucination rate per model and per turn ----------------
# Turn 1 is the same LongFact prompt for every model. Turns 2-3 are follow-ups written from each model's own answers.
data = {}
for name in INPUT_REPOS:
    dst = f"{USER}/{name}-annotated"
    files = sorted(f for f in api.list_repo_files(dst, repo_type="dataset") if f.startswith("data/"))
    rows = [json.loads(l) for f in files for l in open(download(dst, f), encoding="utf-8")]
    data[name] = {(r["conversation_id"], r["turn"]): r for r in rows if r["annotations"] is not None}
common = set.intersection(*(set(d) for d in data.values()))
print(f"{len(common)} assistant turns annotated for every model\n")
print(f"{'model':<46}{'turn':>5}{'entities':>9}{'NS %':>7}{'NS+II %':>9}{'answers w/ >=1 NS %':>21}")
for name, d in data.items():
    for turn in ["all"] + sorted({k[1] for k in common}):
        keys = [k for k in sorted(common) if turn == "all" or k[1] == turn]
        spans = [[a["label"] for a in d[k]["annotations"] if a["label"] and a["index"] is not None] for k in keys]
        flat = [l for s in spans for l in s]
        ns, ii = flat.count("Not Supported"), flat.count("Insufficient Information")
        print(f"{name:<46}{turn:>5}{len(flat):>9}{100 * ns / max(1, len(flat)):>7.1f}{100 * (ns + ii) / max(1, len(flat)):>9.1f}"
              f"{100 * sum('Not Supported' in s for s in spans) / max(1, len(spans)):>21.1f}")